# Notebook 02 — PyTorch Tensors, Autograd, Modules, and Training

    ## Learning objectives

    - Use devices, dtypes, views, modules, datasets, and state dictionaries
- Derive and verify reverse-mode gradients
- Build and resume a correct training loop

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 2.1 Tensors, devices, modules, and state

A tensor combines typed storage, shape and stride metadata, device, and optional gradient history. Move batches once at boundaries and make device/dtype choices explicit. Transposed tensors may be non-contiguous; clone copies, detach stops history, and NumPy conversion is limited to compatible CPU tensors. `nn.Module` recursively registers child modules and Parameters so device movement, optimization, modes, and serialization work. Persistent non-trainable tensors belong in buffers. Inspect named parameters, buffers, unique storage, trainable counts, and optimizer groups before trusting a run.


In [ ]:
import torch
x=torch.arange(24,dtype=torch.float32).reshape(2,3,4); y=x.transpose(1,2)
print(x.stride(),y.stride(),y.is_contiguous(),y.contiguous().view(2,-1).shape)


## 2.2 Reverse-mode automatic differentiation

Autograd records differentiable operations involving tensors that require gradients and propagates vector-Jacobian products backward. It is efficient when one scalar loss depends on many parameters. Gradients accumulate into leaf `.grad`, so reset them at optimizer-update boundaries. In-place operations can invalidate saved intermediates; `.data` bypasses safety checks. Verify custom operations with central finite differences or gradcheck in float64. A missing gradient often indicates detach, NumPy conversion, no-grad construction, an unused parameter, or optimizer exclusion—not a need for arbitrary `retain_graph=True`.


In [ ]:
w=torch.tensor([2.,-1.],requires_grad=True); x=torch.tensor([3.,4.]); loss=((w*x).sum()-5).square(); loss.backward()
print(loss.item(),w.grad)


## 2.3 Data and optimization contracts

Dataset maps indices to records, sampler chooses order, DataLoader fetches, and collator constructs padded or packed batches. Inspect actual decoded batches and label masks. Reduction defines weighting: averaging unequal batch means is not token-weighted loss. The update sequence is forward, correctly reduced loss, backward, optional unscale, gradient clipping, optimizer step, scheduler step, and zeroing. `train()` and `eval()` affect dropout-like modules but do not control gradient recording; use inference mode for evaluation and restore the prior mode.


In [ ]:
from torch import nn
model=nn.Sequential(nn.Linear(4,16),nn.GELU(),nn.Linear(16,3)); opt=torch.optim.AdamW(model.parameters(),lr=1e-3)
a=torch.randn(8,4); target=torch.randint(0,3,(8,)); opt.zero_grad(set_to_none=True); loss=nn.functional.cross_entropy(model(a),target); loss.backward(); print(nn.utils.clip_grad_norm_(model.parameters(),1.0)); opt.step()


## 2.4 Precision, checkpointing, and debugging

FP16 has narrow exponent range, BF16 retains wider range, and autocast selects eligible operations; mixed precision still needs numerical regression tests. Unscale before clipping and log skipped updates. A training checkpoint contains model, optimizer, scheduler, scaler, RNG, step, sampler/data position, and configuration, whereas a deployment artifact may contain only weights and inference metadata. Test a continuous run against a save/resume branch. First overfit one batch and assert finite gradients and changing parameters, then add acceleration, accumulation, checkpointing, compilation, or distribution one axis at a time.


In [ ]:
from pathlib import Path
p=Path("artifacts/prerequisites/model.pt"); p.parent.mkdir(parents=True,exist_ok=True); torch.save({"model":model.state_dict(),"optimizer":opt.state_dict(),"rng":torch.get_rng_state()},p)
restored=type(model)(*list(model.children())); restored.load_state_dict(torch.load(p,map_location="cpu",weights_only=True)["model"]); print("reloaded")


## Reference workflow and evidence standard

Treat the notebook as an experiment, not a recipe. State the question, freeze inputs and
success criteria, establish the simplest baseline, change one material factor, and retain raw
outputs needed to diagnose failures. Record model, tokenizer, template, data and code revisions;
hardware and dtype; random seeds; generation or optimization configuration; token counts;
latency and memory; and results by meaningful slice. A demonstration that runs is evidence of
plumbing, not evidence of general capability.

Test boundaries as well as the happy path: empty and maximum-length inputs, malformed records,
multilingual or code text, unavailable dependencies, cancellation, and adversarial content.
Keep credentials in environment or Colab Secrets and never serialize them with artifacts. Pin
remote revisions, review licenses and custom code, validate saved artifacts in a fresh process,
and prefer deterministic validators wherever outputs can be checked mechanically.

Before applying the technique, compare it with prompting, retrieval, a smaller model, or no
model. Report quality together with compute, storage, latency, and operational complexity. Use
held-out data and paired comparisons, disclose uncertainty and negative results, and define a
rollback path. These practices connect low-level understanding to reliable application work.

A useful completion checklist asks four separate questions. Is the mathematical contract clear
enough to predict shapes, masks, reductions, and failure cases? Does the implementation reproduce
a tiny hand-worked or deterministic reference? Does the measured result survive a held-out set,
relevant slices, and an ablation against a simpler baseline? Can another person reload the exact
artifacts and reconstruct the claim from the manifest? Passing only the first two establishes a
tutorial demonstration; passing all four supports an engineering decision. When a result fails,
preserve the counterexample and update the test suite before changing the implementation.

Finally, separate correctness, capability, efficiency, and safety conclusions. A correct
implementation may have weak capability; a capable prototype may be too costly or unsafe to
deploy. Name the population to which each conclusion applies and avoid converting a single
metric into a universal ranking. Track assumptions beside results, especially tokenizer and
template compatibility, data rights, access-control boundaries, and hardware-specific behavior.
Leave exercises with an executable acceptance criterion, a baseline result, and a short written
interpretation. That combination turns exploratory code into cumulative course evidence that can
be revisited when libraries, model families, or deployment engines change.


## Exercises

    1. Check an analytic gradient with finite differences.
2. Implement token-weighted accumulation.
3. Prove checkpoint resume matches continuous training.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
